In [56]:
import sqlite3
import pandas as pd
from datetime import datetime

# connecties met sdm en dwh databases (brondatabase + dwh database afgeleid van ETL-schema's)

# bron database
sdm_conn = sqlite3.connect("BikeToDriveDatabase.db")

# data warehouse
dwh_conn = sqlite3.connect("DWH_DB.db")

In [55]:
# als er iets mis is met de connection of foutjes in database run dit zodat de connection wordt gestopt en je opnieuw kan proberen !

sdm_conn.close()
dwh_conn.close()

In [61]:
# Extract (combineer beide bronnen)
sdm_klant = pd.read_sql("""
SELECT klantnr, naam, woonplaats, adres, geslacht, geboortedatum, 1 as source_id
FROM Fiets_Verkoop_Klant

UNION ALL

SELECT klantnr, naam, woonplaats, adres, geslacht, geboortedatum, 2 as source_id
FROM Accessoire_Verkoop_Klant
""", sdm_conn)


# Business key maken
sdm_klant['business_key'] = (
    sdm_klant['naam'] + "_" + 
    sdm_klant['geboortedatum'].astype(str)
)


# dubbele waardes eruitgooien binnen SDM
sdm_klant = sdm_klant.drop_duplicates(subset=['business_key'])


# haal bestaande business_keys uit DWH
dwh_keys = pd.read_sql("SELECT business_key FROM Klant", dwh_conn)


# filter alleen nieuwe data
nieuwe_klanten = sdm_klant[
    ~sdm_klant['business_key'].isin(dwh_keys['business_key'])
]


# Insert
for _, row in nieuwe_klanten.iterrows():

    dwh_conn.execute("""
    INSERT INTO Klant (
        business_key, source_id,
        klantnr, naam, woonplaats, adres, geslacht, geboortedatum
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        row['business_key'],
        row['source_id'],
        row['klantnr'],
        row['naam'],
        row['woonplaats'],
        row['adres'],
        row['geslacht'],
        row['geboortedatum']
    ))


dwh_conn.commit()

In [59]:
#verwijder data uit een tabel als je opnieuw wilt proberen
dwh_conn.execute("DELETE FROM Klant")
dwh_conn.commit()

In [45]:
# Extract
sdm_monteur = pd.read_sql("""
SELECT monteurnr, naam, woonplaats, uurloon
FROM Accessoire_Verkoop_Monteur

UNION ALL

SELECT monteurnr, naam, woonplaats, uurloon
FROM Fiets_Verkoop_Monteur
""", sdm_conn)


# Business key
sdm_monteur['business_key'] = (
    sdm_monteur['naam'] + "_" + 
    sdm_monteur['woonplaats']
)


# Deduplicatie
sdm_monteur = sdm_monteur.drop_duplicates(subset=['business_key'])


# Check tegen DWH
dwh_keys = pd.read_sql("SELECT business_key FROM Monteur", dwh_conn)

nieuwe_monteurs = sdm_monteur[
    ~sdm_monteur['business_key'].isin(dwh_keys['business_key'])
]


# Insert (zonder monteur_key!)
for _, row in nieuwe_monteurs.iterrows():
    dwh_conn.execute("""
    INSERT INTO Monteur (
        business_key, monteurnr, naam, woonplaats, uurloon
    )
    VALUES (?, ?, ?, ?, ?)
    """, (
        row['business_key'],
        row['monteurnr'],
        row['naam'],
        row['woonplaats'],
        row['uurloon']
    ))


dwh_conn.commit()

In [20]:
# Extract
sdm_filiaal = pd.read_sql("""
SELECT filiaalnr, naam, adres, provincie
FROM Accessoire_Verkoop_Filiaal

UNION ALL

SELECT filiaalnr, naam, adres, provincie
FROM Fiets_Verkoop_Filiaal
""", sdm_conn)


# Business key
sdm_filiaal['business_key'] = sdm_filiaal['adres']


# Deduplicatie
sdm_filiaal = sdm_filiaal.drop_duplicates(subset=['business_key'])


# Check tegen DWH
dwh_keys = pd.read_sql("SELECT business_key FROM Filiaal", dwh_conn)

nieuwe_filialen = sdm_filiaal[
    ~sdm_filiaal['business_key'].isin(dwh_keys['business_key'])
]


# Insert
for _, row in nieuwe_filialen.iterrows():
    dwh_conn.execute("""
    INSERT INTO Filiaal (
        business_key, filiaalnr, naam, adres, provincie
    )
    VALUES (?, ?, ?, ?, ?)
    """, (
        row['business_key'],
        row['filiaalnr'],
        row['naam'],
        row['adres'],
        row['provincie']
    ))


dwh_conn.commit()

In [21]:
# Extract unieke datums uit SDM
sdm_datum = pd.read_sql("""
SELECT datum FROM Fiets_Verkoop
UNION ALL
SELECT datum FROM Accessoire_Verkoop
""", sdm_conn)

# verwijder duplicates
sdm_datum = sdm_datum.drop_duplicates()

# converteer naar datetime
sdm_datum['datum'] = pd.to_datetime(sdm_datum['datum'])

# maak kolommen
sdm_datum['dag'] = sdm_datum['datum'].dt.day
sdm_datum['maand'] = sdm_datum['datum'].dt.month
sdm_datum['jaar'] = sdm_datum['datum'].dt.year
sdm_datum['kwartaal'] = sdm_datum['datum'].dt.quarter

#  maak datum_key = YYYYMMDD
sdm_datum['datum_key'] = sdm_datum['datum'].dt.strftime('%Y%m%d').astype(int)

# check tegen DWH
dwh_keys = pd.read_sql("SELECT datum_key FROM Datum", dwh_conn)

nieuwe_datums = sdm_datum[
    ~sdm_datum['datum_key'].isin(dwh_keys['datum_key'])
]

# insert
for _, row in sdm_datum.iterrows():
    dwh_conn.execute("""
    INSERT INTO Datum (datum_key, dag, maand, kwartaal, jaar)
    VALUES (?, ?, ?, ?, ?)
    """, (
        row['datum_key'],
        row['dag'],
        row['maand'],
        row['kwartaal'],
        row['jaar']
    ))

dwh_conn.commit()

In [22]:
# Extract
sdm_lev = pd.read_sql("""
SELECT leveranciernr, naam, adres, woonplaats
FROM Accessoire_Verkoop_Leverancier

UNION ALL

SELECT fabrikantnr AS leveranciernr, naam, adres, plaats AS woonplaats
FROM Fiets_Verkoop_Fabrikant
""", sdm_conn)


# Business key
sdm_lev['business_key'] = (
    sdm_lev['naam'] + "_" + 
    sdm_lev['adres']
)


# Deduplicatie
sdm_lev = sdm_lev.drop_duplicates(subset=['business_key'])


# Check tegen DWH
dwh_keys = pd.read_sql("SELECT business_key FROM Leverancier", dwh_conn)

nieuwe_lev = sdm_lev[
    ~sdm_lev['business_key'].isin(dwh_keys['business_key'])
]


# Insert
for _, row in nieuwe_lev.iterrows():
    dwh_conn.execute("""
    INSERT INTO Leverancier (
        business_key, leveranciernr, naam, adres, woonplaats
    )
    VALUES (?, ?, ?, ?, ?)
    """, (
        row['business_key'],
        row['leveranciernr'],
        row['naam'],
        row['adres'],
        row['woonplaats']
    ))


dwh_conn.commit()

In [51]:
# Extract
sdm_product = pd.read_sql("""
SELECT 
    accessoirenr AS productnr,
    'accessoire' AS product_type,
    soort,
    naam AS merk,
    NULL AS type,
    NULL AS kleur,
    leverancier AS fabrikant
FROM Accessoire_Verkoop_Accessoire

UNION ALL

SELECT 
    accessoirenr AS productnr,
    'accessoire' AS product_type,
    soort,
    naam AS merk,
    NULL AS type,
    NULL AS kleur,
    leverancier AS fabrikant
FROM Accessoire_Inkoop_Accessoire

UNION ALL

SELECT 
    fietsnr AS productnr,
    'fiets' AS product_type,
    soort,
    merk,
    type,
    kleur,
    fabrikant
FROM Fiets_Verkoop_Fiets

UNION ALL

SELECT 
    fietsnr AS productnr,
    'fiets' AS product_type,
    soort,
    merk,
    type,
    kleur,
    fabrikant
FROM Fiets_Inkoop_Fiets
""", sdm_conn)


# Business key
sdm_product["business_key"] = (
    sdm_product["product_type"] + "_" +
    sdm_product["productnr"].astype(str)
)


# Deduplicatie
sdm_product = sdm_product.drop_duplicates(subset=["business_key"])


# Check tegen DWH
dwh_keys = pd.read_sql("SELECT business_key FROM Product", dwh_conn)

nieuwe_producten = sdm_product[
    ~sdm_product["business_key"].isin(dwh_keys["business_key"])
].copy()


# Insert
for _, row in nieuwe_producten.iterrows():
    dwh_conn.execute("""
    INSERT INTO Product (
        business_key, productnr, product_type, soort, merk, type, kleur, fabrikant
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        row["business_key"],
        row["productnr"],
        row["product_type"],
        row["soort"],
        row["merk"],
        row["type"],
        row["kleur"],
        row["fabrikant"]
    ))

dwh_conn.commit()

In [ ]:
# Verkoop

# 1. Extract uit SDM
sdm_verkoop = pd.read_sql("""
    SELECT
        fv.fiets_verkoopnr AS verkoopnr,
        fv.datum,
        fv.aantal,
        fv.verkoopprijs,
        fv.klant,
        fv.fiets AS productnr,
        'fiets' AS product_type,
        fv.monteur,

        k.naam AS klant_naam,
        k.geboortedatum,

        m.naam AS monteur_naam,
        m.woonplaats AS monteur_woonplaats,

        f.adres AS filiaal_adres

    FROM Fiets_Verkoop fv
    JOIN Fiets_Verkoop_Klant k
        ON fv.klant = k.klantnr
    JOIN Fiets_Verkoop_Monteur m
        ON fv.monteur = m.monteurnr
    JOIN Fiets_Verkoop_Filiaal f
        ON m.filiaal = f.filiaalnr

    UNION ALL

    SELECT
        av.accessoire_verkoopnr AS verkoopnr,
        av.datum,
        av.aantal,
        av.verkoopprijs,
        av.klant,
        av.accessoire AS productnr,
        'accessoire' AS product_type,
        av.monteur,

        k.naam AS klant_naam,
        k.geboortedatum,

        m.naam AS monteur_naam,
        m.woonplaats AS monteur_woonplaats,

        f.adres AS filiaal_adres

    FROM Accessoire_Verkoop av
    JOIN Accessoire_Verkoop_Klant k
        ON av.klant = k.klantnr
    JOIN Accessoire_Verkoop_Monteur m
        ON av.monteur = m.monteurnr
    JOIN Accessoire_Verkoop_Filiaal f
        ON m.filiaal = f.filiaalnr
""", sdm_conn)


# 2. Datum omzetten naar datum_key
sdm_verkoop["datum"] = pd.to_datetime(sdm_verkoop["datum"])
sdm_verkoop["datum_key"] = sdm_verkoop["datum"].dt.strftime("%Y%m%d").astype(int)


# 3. Business keys maken voor joins
sdm_verkoop["klant_business_key"] = (
    sdm_verkoop["klant_naam"] + "_" +
    sdm_verkoop["geboortedatum"].astype(str)
)

sdm_verkoop["monteur_business_key"] = (
    sdm_verkoop["monteur_naam"] + "_" +
    sdm_verkoop["monteur_woonplaats"]
)

sdm_verkoop["product_business_key"] = (
    sdm_verkoop["product_type"] + "_" +
    sdm_verkoop["productnr"].astype(str)
)

sdm_verkoop["filiaal_business_key"] = sdm_verkoop["filiaal_adres"]


# 4. Dimensies ophalen
klant_dim = pd.read_sql("""
    SELECT klant_key, business_key
    FROM Klant
""", dwh_conn)

monteur_dim = pd.read_sql("""
    SELECT m.monteur_key, m.business_key
    FROM Monteur m
    JOIN (
        SELECT business_key, MAX(monteur_key) AS max_monteur_key
        FROM Monteur
        GROUP BY business_key
    ) x
    ON m.business_key = x.business_key
    AND m.monteur_key = x.max_monteur_key
""", dwh_conn)

product_dim = pd.read_sql("""
    SELECT product_key, business_key
    FROM Product
""", dwh_conn)

filiaal_dim = pd.read_sql("""
    SELECT f.filiaal_key, f.business_key
    FROM Filiaal f
    JOIN (
        SELECT business_key, MAX(filiaal_key) AS max_filiaal_key
        FROM Filiaal
        GROUP BY business_key
    ) x
    ON f.business_key = x.business_key
    AND f.filiaal_key = x.max_filiaal_key
""", dwh_conn)

datum_dim = pd.read_sql("""
    SELECT datum_key
    FROM Datum
""", dwh_conn)


# 5. Join met Klant
sdm_verkoop = sdm_verkoop.merge(
    klant_dim,
    left_on="klant_business_key",
    right_on="business_key",
    how="left"
)
sdm_verkoop = sdm_verkoop.drop(columns=["business_key"])


# 6. Join met Monteur
sdm_verkoop = sdm_verkoop.merge(
    monteur_dim,
    left_on="monteur_business_key",
    right_on="business_key",
    how="left"
)
sdm_verkoop = sdm_verkoop.drop(columns=["business_key"])


# 7. Join met Product
sdm_verkoop = sdm_verkoop.merge(
    product_dim,
    left_on="product_business_key",
    right_on="business_key",
    how="left"
)
sdm_verkoop = sdm_verkoop.drop(columns=["business_key"])


# 8. Join met Filiaal
sdm_verkoop = sdm_verkoop.merge(
    filiaal_dim,
    left_on="filiaal_business_key",
    right_on="business_key",
    how="left"
)
sdm_verkoop = sdm_verkoop.drop(columns=["business_key"])


# 9. Join met Datum
sdm_verkoop = sdm_verkoop.merge(
    datum_dim,
    on="datum_key",
    how="left"
)


# 10. Check op missende keys
missing = sdm_verkoop[
    sdm_verkoop[["klant_key", "monteur_key", "product_key", "filiaal_key", "datum_key"]]
    .isnull()
    .any(axis=1)
]

if not missing.empty:
    print("missende keys in verkoop sukkel")
    print(missing[[
        "verkoopnr",
        "klant_business_key",
        "monteur_business_key",
        "product_business_key",
        "filiaal_business_key",
        "datum_key",
        "klant_key",
        "monteur_key",
        "product_key",
        "filiaal_key"
    ]].head())
else:
    print("Geen missende keys in Verkoop 🙄")


# 11. Fact tabel maken
fact_verkoop = sdm_verkoop[
    ["klant_key", "monteur_key", "product_key", "filiaal_key", "datum_key", "verkoopprijs", "aantal"]
].copy()


# 12. Rijen zonder geldige keys verwijderen
fact_verkoop = fact_verkoop.dropna(
    subset=["klant_key", "monteur_key", "product_key", "filiaal_key", "datum_key"]
)


# 13. Types netjes zetten
fact_verkoop["klant_key"] = fact_verkoop["klant_key"].astype(int)
fact_verkoop["monteur_key"] = fact_verkoop["monteur_key"].astype(int)
fact_verkoop["product_key"] = fact_verkoop["product_key"].astype(int)
fact_verkoop["filiaal_key"] = fact_verkoop["filiaal_key"].astype(int)
fact_verkoop["datum_key"] = fact_verkoop["datum_key"].astype(int)


# 14. Laden naar DWH
fact_verkoop.to_sql(
    "Verkoop",
    dwh_conn,
    if_exists="append",
    index=False
)


# 15. Check
print(fact_verkoop.head())

✅ Geen missende keys in Verkoop
   klant_key  monteur_key  product_key  filiaal_key  datum_key  verkoopprijs  \
0         25            6           74            3   20240309       1020.48   
1         23            5           71            3   20240712       1391.84   
2          2           10           58            2   20241121       1891.33   
3          9            5           42            3   20240526        995.53   
4         13           10           51            2   20240415       1568.90   

   aantal  
0       3  
1       1  
2       2  
3       3  
4       2  


In [ ]:
#Onderhoud

# 1. Extract uit SDM
sdm_onderhoud = pd.read_sql("""
    SELECT
        o.onderhoudnr,
        o.datum,
        o.starttijd,
        o.eindtijd,
        o.fiets AS productnr,
        'fiets' AS product_type,
        o.monteur,

        m.naam AS monteur_naam,
        m.woonplaats AS monteur_woonplaats,
        m.uurloon,
        m.filiaal,

        f.adres AS filiaal_adres

    FROM Onderhoud o
    JOIN Onderhoud_Monteur m
        ON o.monteur = m.monteurnr
    JOIN Onderhoud_Filiaal f
        ON m.filiaal = f.filiaalnr
""", sdm_conn)


# 2. Business keys maken
sdm_onderhoud["product_business_key"] = (
    sdm_onderhoud["product_type"] + "_" +
    sdm_onderhoud["productnr"].astype(str)
)

sdm_onderhoud["monteur_business_key"] = (
    sdm_onderhoud["monteur_naam"] + "_" +
    sdm_onderhoud["monteur_woonplaats"]
)

sdm_onderhoud["filiaal_business_key"] = sdm_onderhoud["filiaal_adres"]


# 3. Dimensies ophalen
product_dim = pd.read_sql("""
    SELECT product_key, business_key
    FROM Product
""", dwh_conn)

monteur_dim = pd.read_sql("""
    SELECT m.monteur_key, m.business_key
    FROM Monteur m
    JOIN (
        SELECT business_key, MAX(monteur_key) AS max_monteur_key
        FROM Monteur
        GROUP BY business_key
    ) x
    ON m.business_key = x.business_key
    AND m.monteur_key = x.max_monteur_key
""", dwh_conn)

filiaal_dim = pd.read_sql("""
    SELECT f.filiaal_key, f.business_key
    FROM Filiaal f
    JOIN (
        SELECT business_key, MAX(filiaal_key) AS max_filiaal_key
        FROM Filiaal
        GROUP BY business_key
    ) x
    ON f.business_key = x.business_key
    AND f.filiaal_key = x.max_filiaal_key
""", dwh_conn)


# 4. Join met Product
sdm_onderhoud = sdm_onderhoud.merge(
    product_dim,
    left_on="product_business_key",
    right_on="business_key",
    how="left"
)
sdm_onderhoud = sdm_onderhoud.drop(columns=["business_key"])


# 5. Join met Monteur
sdm_onderhoud = sdm_onderhoud.merge(
    monteur_dim,
    left_on="monteur_business_key",
    right_on="business_key",
    how="left"
)
sdm_onderhoud = sdm_onderhoud.drop(columns=["business_key"])


# 6. Join met Filiaal
sdm_onderhoud = sdm_onderhoud.merge(
    filiaal_dim,
    left_on="filiaal_business_key",
    right_on="business_key",
    how="left"
)
sdm_onderhoud = sdm_onderhoud.drop(columns=["business_key"])


# 7. Check op missende keys
missing = sdm_onderhoud[
    sdm_onderhoud[["product_key", "monteur_key", "filiaal_key"]]
    .isnull()
    .any(axis=1)
]

if not missing.empty:
    print("WAARSCHUWING: missende keys in onderhoud pur slay queen")
    print(missing[[
        "onderhoudnr",
        "product_business_key",
        "monteur_business_key",
        "filiaal_business_key",
        "product_key",
        "monteur_key",
        "filiaal_key"
    ]].head())
else:
    print("Geen missende keys in Onderhoud 😛😛😛😛")


# 8. Fact tabel maken
fact_onderhoud = sdm_onderhoud[
    ["product_key", "monteur_key", "filiaal_key", "datum", "starttijd", "eindtijd", "uurloon"]
].copy()


# 9. Rijen zonder geldige keys verwijderen
fact_onderhoud = fact_onderhoud.dropna(
    subset=["product_key", "monteur_key", "filiaal_key"]
)


# 10. Types netjes zetten
fact_onderhoud["product_key"] = fact_onderhoud["product_key"].astype(int)
fact_onderhoud["monteur_key"] = fact_onderhoud["monteur_key"].astype(int)
fact_onderhoud["filiaal_key"] = fact_onderhoud["filiaal_key"].astype(int)


# 11. Laden naar DWH
fact_onderhoud.to_sql(
    "Onderhoud",
    dwh_conn,
    if_exists="append",
    index=False
)


# 12. Check
print(fact_onderhoud.head())

⚠️ WARNING: missende keys in Onderhoud
   onderhoudnr product_business_key    monteur_business_key  \
0            1             fiets_22     Pim Scholten_Leiden   
2            3              fiets_3     Pim Scholten_Leiden   
4            5             fiets_60   Tijn Hendriks_Zaandam   
5            6             fiets_18  Laura Smeets_Amsterdam   
8            9             fiets_18    Nick Smits_Rotterdam   

  filiaal_business_key  product_key  monteur_key  filiaal_key  
0          Damstraat 5           35          NaN          NaN  
2          Damstraat 5           16          NaN          NaN  
4          Damstraat 5           73          NaN          NaN  
5    Prinsengracht 100           31          NaN          1.0  
8    Prinsengracht 100           31          NaN          1.0  
   product_key  monteur_key  filiaal_key       datum         starttijd  \
1           31            8            4  2024-02-05  14:00:00.0000000   
3           24            2            1  2024-11-

In [ ]:
#Inkoop

# 1. Extract uit SDM
sdm_inkoop = pd.read_sql("""
    SELECT 
        ai.inkoopnr,
        ai.inkoopmaand,
        ai.inkoopjaar,
        ai.aantal,
        a.accessoirenr AS productnr,
        'accessoire' AS product_type,
        l.leveranciernr,
        l.naam AS leverancier_naam,
        l.adres AS leverancier_adres,
        a.inkoopprijs
    FROM Accessoire_Inkoop ai
    JOIN Accessoire_Inkoop_Accessoire a 
        ON ai.accessoire = a.accessoirenr
    JOIN Accessoire_Inkoop_Leverancier l 
        ON a.leverancier = l.leveranciernr

    UNION ALL

    SELECT 
        fi.inkoopnr,
        fi.inkoopmaand,
        fi.inkoopjaar,
        fi.aantal,
        f.fietsnr AS productnr,
        'fiets' AS product_type,
        fab.fabrikantnr AS leveranciernr,
        fab.naam AS leverancier_naam,
        fab.adres AS leverancier_adres,
        f.inkoopprijs
    FROM Fiets_Inkoop fi
    JOIN Fiets_Inkoop_Fiets f 
        ON fi.fiets = f.fietsnr
    JOIN Fiets_Inkoop_Fabrikant fab 
        ON f.fabrikant = fab.fabrikantnr
""", sdm_conn)


# 2. Business keys maken voor joins
# Product gebruikt in jouw dimensie: product_type + "_" + productnr
sdm_inkoop["product_business_key"] = (
    sdm_inkoop["product_type"] + "_" +
    sdm_inkoop["productnr"].astype(str)
)

# Leverancier gebruikt in jouw dimensie: naam + "_" + adres
sdm_inkoop["lev_business_key"] = (
    sdm_inkoop["leverancier_naam"] + "_" +
    sdm_inkoop["leverancier_adres"]
)


# 3. Dimensies ophalen
product_dim = pd.read_sql("""
    SELECT product_key, business_key
    FROM Product
""", dwh_conn)

periode_dim = pd.read_sql("""
    SELECT periode_key, inkoopmaand, inkoopjaar
    FROM InkoopPeriode
""", dwh_conn)

leverancier_dim = pd.read_sql("""
    SELECT lev_key, business_key
    FROM Leverancier
""", dwh_conn)


# 4. Join met Product dimensie
sdm_inkoop = sdm_inkoop.merge(
    product_dim,
    left_on="product_business_key",
    right_on="business_key",
    how="left"
)

# extra business_key kolom van merge weer weg
sdm_inkoop = sdm_inkoop.drop(columns=["business_key"])


# 5. Join met InkoopPeriode dimensie
sdm_inkoop = sdm_inkoop.merge(
    periode_dim,
    on=["inkoopmaand", "inkoopjaar"],
    how="left"
)


# 6. Join met Leverancier dimensie
sdm_inkoop = sdm_inkoop.merge(
    leverancier_dim,
    left_on="lev_business_key",
    right_on="business_key",
    how="left"
)

# extra business_key kolom van merge weer weg
sdm_inkoop = sdm_inkoop.drop(columns=["business_key"])


# 7. Check op missende keys
missing = sdm_inkoop[
    sdm_inkoop[["product_key", "periode_key", "lev_key"]].isnull().any(axis=1)
]

if not missing.empty:
    print("WAARSCHUWING: missende keys")
    print(missing[[
        "inkoopnr",
        "productnr",
        "product_type",
        "leveranciernr",
        "leverancier_naam",
        "leverancier_adres",
        "product_business_key",
        "lev_business_key",
        "product_key",
        "periode_key",
        "lev_key"
    ]].head())


# 8. Fact tabel maken
fact_inkoop = sdm_inkoop[
    ["inkoopnr", "product_key", "periode_key", "lev_key", "aantal", "inkoopprijs"]
].copy()


# 9. Rijen zonder geldige foreign keys verwijderen
fact_inkoop = fact_inkoop.dropna(
    subset=["product_key", "periode_key", "lev_key"]
)


# 10. Optioneel: integer kolommen netjes casten
fact_inkoop["product_key"] = fact_inkoop["product_key"].astype(int)
fact_inkoop["periode_key"] = fact_inkoop["periode_key"].astype(int)
fact_inkoop["lev_key"] = fact_inkoop["lev_key"].astype(int)


# 11. Laden naar DWH
fact_inkoop.to_sql(
    "Inkoop",
    dwh_conn,
    if_exists="append",
    index=False
)


# 12. Check
print(fact_inkoop.head())

   inkoopnr  product_key  periode_key  lev_key  aantal  inkoopprijs
0         1            7            4        2      38        24.90
1         2            6            8        2      31        18.75
2         3            2            1        1      41         7.25
3         4            1            1        1      21         8.50
4         5            4           10        1      31        11.40


In [25]:
# InkoopPeriode

sdm_inkoopperiode = pd.read_sql("""
SELECT inkoopmaand, inkoopjaar
FROM Fiets_Inkoop

UNION ALL

SELECT inkoopmaand, inkoopjaar
FROM Accessoire_Inkoop
""", sdm_conn)

# dubbele maand-jaar combinaties weg
sdm_inkoopperiode = sdm_inkoopperiode.drop_duplicates()

# netjes sorteren
sdm_inkoopperiode = sdm_inkoopperiode.sort_values(
    by=["inkoopjaar", "inkoopmaand"]
).reset_index(drop=True)

# bestaande periodes uit DWH halen
dwh_periodes = pd.read_sql(
    "SELECT inkoopmaand, inkoopjaar FROM InkoopPeriode",
    dwh_conn
)

# alleen nieuwe periodes bepalen
nieuwe_periodes = sdm_inkoopperiode.merge(
    dwh_periodes,
    on=["inkoopmaand", "inkoopjaar"],
    how="left",
    indicator=True
).query('_merge == "left_only"').drop(columns=['_merge'])

# startwaarde surrogate key bepalen
start_key = pd.read_sql(
    "SELECT COALESCE(MAX(periode_key), 0) as max_key FROM InkoopPeriode",
    dwh_conn
)['max_key'][0]

# surrogate key maken
nieuwe_periodes["periode_key"] = range(
    start_key + 1,
    start_key + 1 + len(nieuwe_periodes)
)

# juiste kolomvolgorde
nieuwe_periodes = nieuwe_periodes[
    ["periode_key", "inkoopmaand", "inkoopjaar"]
]

# in DWH zetten
nieuwe_periodes.to_sql(
    "InkoopPeriode",
    dwh_conn,
    if_exists="append",
    index=False
)

print(nieuwe_periodes)

    periode_key  inkoopmaand  inkoopjaar
0             1            1        2024
1             2            2        2024
2             3            3        2024
3             4            4        2024
4             5            5        2024
5             6            6        2024
6             7            7        2024
7             8            8        2024
8             9            9        2024
9            10           10        2024
10           11           11        2024
11           12           12        2024


In [ ]:
# scd type 2 voor Monteur

# 1. Extract
sdm_monteur = pd.read_sql("""
SELECT monteurnr, naam, woonplaats, uurloon
FROM Accessoire_Verkoop_Monteur

UNION ALL

SELECT monteurnr, naam, woonplaats, uurloon
FROM Fiets_Verkoop_Monteur
""", sdm_conn)

# 2. Business key maken
sdm_monteur["business_key"] = (
    sdm_monteur["naam"] + "_" +
    sdm_monteur["woonplaats"]
)

# 3. Duplicaten verwijderen
sdm_monteur = sdm_monteur.drop_duplicates(subset=["business_key"])

# 4. Alleen meest actuele rij per business key uit DWH halen
# actueel = hoogste surrogate key
dwh_monteur = pd.read_sql("""
SELECT m.*
FROM Monteur m
JOIN (
    SELECT business_key, MAX(monteur_key) AS max_monteur_key
    FROM Monteur
    GROUP BY business_key
) x
ON m.business_key = x.business_key
AND m.monteur_key = x.max_monteur_key
""", dwh_conn)

# 5. SDM vergelijken met actuele DWH-data
merged = sdm_monteur.merge(
    dwh_monteur,
    on="business_key",
    how="left",
    suffixes=("_sdm", "_dwh")
)

# 6. Nieuwe of gewijzigde rijen toevoegen
for _, row in merged.iterrows():

    # bestaat nog niet in DWH -> insert
    if pd.isna(row["monteur_key"]):
        dwh_conn.execute("""
        INSERT INTO Monteur (
            business_key, monteurnr, naam, woonplaats, uurloon
        )
        VALUES (?, ?, ?, ?, ?)
        """, (
            row["business_key"],
            row["monteurnr_sdm"],
            row["naam_sdm"],
            row["woonplaats_sdm"],
            row["uurloon_sdm"]
        ))

    else:
        # check of iets veranderd is
        gewijzigd = (
            row["monteurnr_sdm"] != row["monteurnr_dwh"] or
            row["naam_sdm"] != row["naam_dwh"] or
            row["woonplaats_sdm"] != row["woonplaats_dwh"] or
            row["uurloon_sdm"] != row["uurloon_dwh"]
        )

        # bij wijziging: nieuwe rij inserten
        if gewijzigd:
            dwh_conn.execute("""
            INSERT INTO Monteur (
                business_key, monteurnr, naam, woonplaats, uurloon
            )
            VALUES (?, ?, ?, ?, ?)
            """, (
                row["business_key"],
                row["monteurnr_sdm"],
                row["naam_sdm"],
                row["woonplaats_sdm"],
                row["uurloon_sdm"]
            ))

# 7. Commit
dwh_conn.commit()

print("Monteur SCD Type 2 klaar")

In [ ]:
# scd type 2 voor Filiaal


# 1. Extract
sdm_filiaal = pd.read_sql("""
SELECT filiaalnr, naam, adres, provincie
FROM Accessoire_Verkoop_Filiaal

UNION ALL

SELECT filiaalnr, naam, adres, provincie
FROM Fiets_Verkoop_Filiaal
""", sdm_conn)

# 2. Business key maken
sdm_filiaal["business_key"] = sdm_filiaal["adres"]

# 3. Duplicaten verwijderen
sdm_filiaal = sdm_filiaal.drop_duplicates(subset=["business_key"])

# 4. Alleen meest actuele rij per business key uit DWH halen
# actueel = hoogste surrogate key
dwh_filiaal = pd.read_sql("""
SELECT f.*
FROM Filiaal f
JOIN (
    SELECT business_key, MAX(filiaal_key) AS max_filiaal_key
    FROM Filiaal
    GROUP BY business_key
) x
ON f.business_key = x.business_key
AND f.filiaal_key = x.max_filiaal_key
""", dwh_conn)

# 5. SDM vergelijken met actuele DWH-data
merged = sdm_filiaal.merge(
    dwh_filiaal,
    on="business_key",
    how="left",
    suffixes=("_sdm", "_dwh")
)

# 6. Nieuwe of gewijzigde rijen toevoegen
for _, row in merged.iterrows():

    # bestaat nog niet in DWH -> insert
    if pd.isna(row["filiaal_key"]):
        dwh_conn.execute("""
        INSERT INTO Filiaal (
            business_key, filiaalnr, naam, adres, provincie
        )
        VALUES (?, ?, ?, ?, ?)
        """, (
            row["business_key"],
            row["filiaalnr_sdm"],
            row["naam_sdm"],
            row["adres_sdm"],
            row["provincie_sdm"]
        ))

    else:
        # check of iets veranderd is
        gewijzigd = (
            row["filiaalnr_sdm"] != row["filiaalnr_dwh"] or
            row["naam_sdm"] != row["naam_dwh"] or
            row["adres_sdm"] != row["adres_dwh"] or
            row["provincie_sdm"] != row["provincie_dwh"]
        )

        # bij wijziging: nieuwe rij inserten
        if gewijzigd:
            dwh_conn.execute("""
            INSERT INTO Filiaal (
                business_key, filiaalnr, naam, adres, provincie
            )
            VALUES (?, ?, ?, ?, ?)
            """, (
                row["business_key"],
                row["filiaalnr_sdm"],
                row["naam_sdm"],
                row["adres_sdm"],
                row["provincie_sdm"]
            ))

# 7. Commit
dwh_conn.commit()

print("Filiaal SCD Type 2 klaar")